# Notebook 02b — Espansione TMDb completa, incrementale e riprendibile

Variante **senza modificare il notebook 02 originale**. Riusa tutti i `tmdb_raw_*.jsonl` già presenti; non riscarica film validi, salva i nuovi JSON immediatamente e crea un dataset **cumulativo** preservando l'intero catalogo MovieLens.

- **Prima esecuzione:** `RUN_FETCH=False` per controllare quanti film sono già nella cache.
- **Test 50 nuovi film:** `RUN_FETCH=True`, `MAX_NEW_MOVIES=50`.
- **Esecuzione completa:** `RUN_FETCH=True`, `MAX_NEW_MOVIES=None`. Si può interrompere e riprendere.
- **Nessuna credenziale inclusa:** utilizza `.env` con `TMDB_API_ACCESS_TOKEN`.
- Output: `dataset_enriched_movielens_tmdb_full.{csv,parquet}`; il vecchio dataset non viene sovrascritto.


#### NOTA:
preferisco mantenere retrocompatibilità piuttosto che dover gestire gte_text_no_ml_tags e portare rischi


In [11]:
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional
import json, os, time, logging
import pandas as pd
import numpy as np
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
logger = logging.getLogger('tmdb-incremental')

# ---- CONFIGURAZIONE ----
RUN_FETCH = True            # impostare True SOLO quando si desidera chiamare TMDb
MAX_NEW_MOVIES = None         # None = tutti i nuovi, 50 = prova incrementale
RATE_LIMIT_DELAY = 0.35     # pausa tra richieste dell'applicazione (i retry possono aggiungere altre pause)
REQUEST_TIMEOUT = 20
MAX_RETRIES = 4
MIN_CACHE_COVERAGE_FOR_EXPORT = 0.97

root_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in root_candidates if (p / 'datasets' / 'raw' / 'ml-latest-small' / 'movies.csv').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cercato datasets/raw/ml-latest-small/movies.csv nella cartella corrente o parent.')
DATA_RAW = PROJECT_ROOT / 'datasets' / 'raw' / 'ml-latest-small'
DATA_PROCESSED = PROJECT_ROOT / 'datasets' / 'processed'
CHECKPOINTS_DIR = DATA_PROCESSED / 'checkpoints'
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT / '.env')
TOKEN = os.getenv('TMDB_API_ACCESS_TOKEN')
print('Project root:', PROJECT_ROOT)
print('RUN_FETCH:', RUN_FETCH, '| MAX_NEW_MOVIES:', MAX_NEW_MOVIES)

Project root: /Users/danieleraveglia/Documents/uniMi/TESI/progetto
RUN_FETCH: True | MAX_NEW_MOVIES: None


## 1. Catalogo completo MovieLens — 9.742 film attesi

Il notebook legge `movies.csv`/`links.csv` **senza escludere film che non hanno rating**. Non serve rieseguire il notebook 01. `movieId` rimane la chiave primaria: il `tmdbId` identifica il metadato esterno e può essere riutilizzato in più righe MovieLens.

In [3]:
movies = pd.read_csv(DATA_RAW / 'movies.csv')
links = pd.read_csv(DATA_RAW / 'links.csv', dtype={'imdbId': 'string', 'tmdbId': 'Int64'})
ratings = pd.read_csv(DATA_RAW / 'ratings.csv')
tags = pd.read_csv(DATA_RAW / 'tags.csv')

assert movies.movieId.is_unique
assert links.movieId.is_unique
ml_base = movies.merge(links, on='movieId', how='left', validate='one_to_one')
ml_base['tmdbId'] = pd.to_numeric(ml_base['tmdbId'], errors='coerce').astype('Int64')
# stats necessarie anche per compatibilità con lo schema del notebook originale
stats = ratings.groupby('movieId').rating.agg(rating_count='size', avg_rating='mean').reset_index()
ml_base = ml_base.merge(stats, on='movieId', how='left', validate='one_to_one')
ml_base['rating_count'] = ml_base['rating_count'].fillna(0).astype(int)
df_to_fetch = ml_base.copy()  # nome compatibile con merge originale del notebook 02
print('Catalogo:', len(ml_base))
print('Con tmdbId:', ml_base.tmdbId.notna().sum())
print('Senza tmdbId:', ml_base.tmdbId.isna().sum())
print('Senza rating:', (ml_base.rating_count == 0).sum())

Catalogo: 9742
Con tmdbId: 9734
Senza tmdbId: 8
Senza rating: 18


## 2. Cache globale da *tutti* i JSONL storici

La cache non è soltanto l'ultimo file. Tutte le sessioni vengono ricostruite e deduplicate per `tmdbId`. Se un JSONL parziale è stato salvato prima di un'interruzione, viene letto normalmente alla ripartenza.

In [4]:
def load_all_tmdb_checkpoints(folder: Path) -> tuple[dict[int,dict], list[dict]]:
    records = {}
    issues = []
    files = sorted(folder.glob('tmdb_raw_*.jsonl'))
    for file in files:
        with file.open('r', encoding='utf-8') as handle:
            for line_number, line in enumerate(handle, 1):
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                    if not isinstance(obj, dict) or not isinstance(obj.get('id'), int):
                        raise ValueError('Record senza id TMDb intero')
                    tmdb_id = int(obj['id'])
                    if 'tmdbId' in obj and int(obj['tmdbId']) != tmdb_id:
                        raise ValueError('Mismatch tra tmdbId e id')
                    # La variante più recente valida vince; gli altri file non vengono modificati.
                    records[tmdb_id] = obj
                except Exception as e:
                    issues.append({'file': file.name, 'line': line_number, 'error': str(e)})
    print(f'Checkpoint: {len(files)} file, {len(records)} TMDb ID unici, {len(issues)} righe problematiche')
    if issues:
        print('Esempio righe problematiche:', issues[:3])
    return records, issues

tmdb_raw_data, cache_issues = load_all_tmdb_checkpoints(CHECKPOINTS_DIR)
requested_ids = set(ml_base.tmdbId.dropna().astype(int))
missing_ids = sorted(requested_ids - set(tmdb_raw_data))
print('Già disponibili:', len(requested_ids & set(tmdb_raw_data)))
print('Ancora da recuperare:', len(missing_ids))

Checkpoint: 3 file, 1996 TMDb ID unici, 0 righe problematiche
Già disponibili: 1996
Ancora da recuperare: 7737


## 3. Recupero opzionale degli otto tmdbId mancanti via IMDb

L'ID numerico MovieLens viene normalizzato, ad es. `114709` → `tt0114709`. La cache delle risoluzioni è separata dal cache dei dettagli e viene riutilizzata. Un eventuale mancato abbinamento non elimina il film dal catalogo.

In [12]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

class TMDbClient:
    def __init__(self, token: str):
        self.session = requests.Session()
        self.session.headers.update({'Authorization': f'Bearer {token}', 'Accept': 'application/json'})
        strategy = Retry(total=MAX_RETRIES, status_forcelist=[429,500,502,503,504],
                         backoff_factor=1.5, respect_retry_after_header=True,
                         allowed_methods=['GET'])
        self.session.mount('https://', HTTPAdapter(max_retries=strategy))
        self.last_request = 0.0

    def get(self, endpoint: str, params: Optional[dict] = None) -> dict:
        sleep_for = RATE_LIMIT_DELAY - (time.monotonic() - self.last_request)
        if sleep_for > 0: time.sleep(sleep_for)
        self.last_request = time.monotonic()
        response = self.session.get('https://api.themoviedb.org/3' + endpoint,
                                    params=params, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()       # distingue 404, 429, 5xx e problemi di rete
        return response.json()

    def close(self):
        self.session.close()

def imdb_to_external_id(v):
    if pd.isna(v): return None
    digits = str(v).strip()
    if digits.startswith('tt'): digits = digits[2:]
    if not digits.isdigit(): return None
    return 'tt' + digits.zfill(7)

resolution_path = CHECKPOINTS_DIR / 'imdb_tmdb_resolutions.json'
resolution_cache = json.loads(resolution_path.read_text(encoding='utf-8')) if resolution_path.exists() else {}

client = None
if RUN_FETCH:
    if not TOKEN: raise ValueError('TMDB_API_ACCESS_TOKEN non presente nel file .env')
    client = TMDbClient(TOKEN)

unresolved_rows = ml_base.index[ml_base.tmdbId.isna()].tolist()
for idx in unresolved_rows:
    external_id = imdb_to_external_id(ml_base.at[idx, 'imdbId'])
    if not external_id: continue
    if external_id in resolution_cache:
        result_id = resolution_cache[external_id]
    elif not RUN_FETCH:
        continue
    else:
        try:
            response = client.get(f'/find/{external_id}', {'external_source': 'imdb_id'})
            match = (response.get('movie_results') or [None])[0]
            result_id = int(match['id']) if match else None
            resolution_cache[external_id] = result_id
            tmp_path = resolution_path.with_suffix('.json.tmp')
            tmp_path.write_text(json.dumps(resolution_cache, indent=2), encoding='utf-8')
            tmp_path.replace(resolution_path)
        except requests.RequestException as exc:
            logger.warning('IMDb lookup %s non riuscito: %s', external_id, exc)
            continue
    if result_id is not None:
        ml_base.at[idx, 'tmdbId'] = int(result_id)

ml_base['tmdbId'] = ml_base['tmdbId'].astype('Int64')
df_to_fetch = ml_base.copy()
requested_ids = set(ml_base.tmdbId.dropna().astype(int))
missing_ids = sorted(requested_ids - set(tmdb_raw_data))
print('Dopo IMDb fallback, nuovi TMDb ID da cercare:', len(missing_ids))

Dopo IMDb fallback, nuovi TMDb ID da cercare: 7690


## 4. Download incrementale con scrittura dopo *ogni film*

I nuovi risultati vengono aggiunti direttamente a un JSONL della sessione. In caso di interruzione il file resta riutilizzabile. I fallimenti finiscono nel log, ma non vengono memorizzati come successi. Una successiva esecuzione riprova gli ID mancanti.

In [14]:
new_ids = missing_ids if MAX_NEW_MOVIES is None else missing_ids[:int(MAX_NEW_MOVIES)]
session_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
new_checkpoint = CHECKPOINTS_DIR / f'tmdb_raw_{session_timestamp}.jsonl'
fetch_errors = []
succeeded = 0

if RUN_FETCH:
    assert client is not None
    with new_checkpoint.open('a', encoding='utf-8') as out:
        for i, tmdb_id in enumerate(new_ids, 1):
            try:
                payload = client.get(f'/movie/{tmdb_id}', {'append_to_response': 'credits,keywords'})
                if int(payload.get('id', -1)) != int(tmdb_id):
                    raise ValueError(f'ID mismatch per {tmdb_id}')
                obj = {'tmdbId': int(tmdb_id), **payload}
                out.write(json.dumps(obj, ensure_ascii=False) + '\n')
                out.flush()  # il record rimane disponibile anche se la cella viene interrotta
                tmdb_raw_data[int(tmdb_id)] = obj
                succeeded += 1
            except (requests.RequestException, ValueError) as exc:
                fetch_errors.append({'tmdbId': tmdb_id, 'error': str(exc)})
            if i % 50 == 0 or i == len(new_ids):
                print(f'{i}/{len(new_ids)} richiesti | +{succeeded} salvati | {len(fetch_errors)} errori')
    if fetch_errors:
        err_file = CHECKPOINTS_DIR / f'fetch_errors_{session_timestamp}.json'
        err_file.write_text(json.dumps(fetch_errors, ensure_ascii=False, indent=2), encoding='utf-8')
    client.close()
else:
    print('DRY RUN: nessuna chiamata TMDb. Impostare RUN_FETCH=True per scaricare.')

print('Film TMDb presenti complessivamente in cache:', len(tmdb_raw_data))
print('Nuovi salvati in questa sessione:', succeeded)

50/7690 richiesti | +50 salvati | 0 errori
100/7690 richiesti | +100 salvati | 0 errori


KeyboardInterrupt: 

## 5. Parsing — riutilizza la funzione canonica del notebook 02

La funzione che segue è copiata dal codice esistente, in modo da preservare i campi e non inventare un nuovo schema. Il parsing viene eseguito sull'**unione di tutte le sessioni**, non soltanto sui nuovi film.

In [15]:
print("\n" + "="*80)
print("PARSING E NORMALIZZAZIONE DATI TMDb (SCHEMA CANONICO)")
print("="*80)

def _safe_join(values: List[str], sep: str = ", ") -> str:
    cleaned = [str(v).strip() for v in values if isinstance(v, str) and str(v).strip()]
    return sep.join(cleaned)


def extract_tmdb_metadata(raw_data: Dict) -> Dict:
    """
    Estrae metadata strutturati da risposta TMDb secondo schema canonico.
    """
    genres = [g.get('name', '') for g in raw_data.get('genres', []) if isinstance(g, dict)]

    origin_countries = raw_data.get('origin_country') or []
    production_countries = [c.get('iso_3166_1', '') for c in raw_data.get('production_countries', []) if isinstance(c, dict)]
    spoken_languages = [l.get('iso_639_1', '') for l in raw_data.get('spoken_languages', []) if isinstance(l, dict)]

    cast_list = raw_data.get('credits', {}).get('cast', []) if isinstance(raw_data.get('credits'), dict) else []
    crew_list = raw_data.get('credits', {}).get('crew', []) if isinstance(raw_data.get('credits'), dict) else []

    directors = [c.get('name', '') for c in crew_list if c.get('job') == 'Director']
    writers = [c.get('name', '') for c in crew_list if c.get('job') in {'Writer', 'Screenplay'}]
    composers = [c.get('name', '') for c in crew_list if c.get('job') in {'Original Music Composer', 'Music'}]

    cast_top = [c.get('name', '') for c in cast_list[:5]]
    cast_characters_top = [c.get('character', '') for c in cast_list[:5]]

    keywords_raw = raw_data.get('keywords', {})
    keyword_items = keywords_raw.get('keywords', []) if isinstance(keywords_raw, dict) else []
    keyword_names = [k.get('name', '') for k in keyword_items if isinstance(k, dict)]

    production_companies = [p.get('name', '') for p in raw_data.get('production_companies', []) if isinstance(p, dict)]

    release_date = raw_data.get('release_date', '') or ''
    release_year = None
    if release_date and len(release_date) >= 4:
        try:
            release_year = int(release_date[:4])
        except ValueError:
            release_year = None

    overview = raw_data.get('overview', '') or ''
    tagline = raw_data.get('tagline', '') or ''
    tmdb_text = ' '.join(part for part in [overview, tagline, _safe_join(keyword_names, ' ')] if part).strip()

    extracted = {
        'tmdbId': raw_data.get('id'),
        'imdbId': raw_data.get('imdb_id'),
        'tmdb_title': raw_data.get('title', ''),
        'tmdb_original_title': raw_data.get('original_title', ''),
        'release_date': release_date,
        'release_year': release_year,
        'tmdb_genres': _safe_join(genres),
        'tmdb_keywords': _safe_join(keyword_names),
        'tmdb_keyword_text': _safe_join(keyword_names, ' '),
        'tmdb_keyword_count': len(keyword_names),
        'overview': overview,
        'tagline': tagline,
        'runtime': raw_data.get('runtime'),
        'original_language': raw_data.get('original_language'),
        'origin_countries': _safe_join(origin_countries),
        'production_countries': _safe_join(production_countries),
        'spoken_languages': _safe_join(spoken_languages),
        'adult': raw_data.get('adult'),
        'directors': _safe_join(directors),
        'writers': _safe_join(writers),
        'composers': _safe_join(composers),
        'cast_top': _safe_join(cast_top),
        'cast_characters_top': _safe_join(cast_characters_top),
        'collection_name': (raw_data.get('belongs_to_collection') or {}).get('name') if isinstance(raw_data.get('belongs_to_collection'), dict) else None,
        'production_companies': _safe_join(production_companies),
        'tmdb_vote_average': raw_data.get('vote_average'),
        'tmdb_vote_count': raw_data.get('vote_count'),
        'tmdb_popularity': raw_data.get('popularity'),
        'budget': raw_data.get('budget'),
        'revenue': raw_data.get('revenue'),
        'poster_path': raw_data.get('poster_path'),
        'backdrop_path': raw_data.get('backdrop_path'),
        'homepage': raw_data.get('homepage'),
        'tmdb_enrichment_status': 'ok',
        'tmdb_retrieved_at': datetime.now().isoformat(timespec='seconds'),
        'tmdb_text': tmdb_text,
    }
    return extracted


PARSING E NORMALIZZAZIONE DATI TMDb (SCHEMA CANONICO)


In [16]:
parsed_metadata = {}
for tmdb_id in requested_ids & set(tmdb_raw_data):
    raw = tmdb_raw_data[tmdb_id]
    try:
        parsed_metadata[tmdb_id] = extract_tmdb_metadata(raw)
    except Exception as exc:
        logger.error('Parsing fallito per %s: %s', tmdb_id, exc)

df_tmdb_enriched = pd.DataFrame.from_dict(parsed_metadata, orient='index')
if df_tmdb_enriched.empty:
    df_tmdb_enriched = pd.DataFrame(columns=['tmdbId'])
else:
    df_tmdb_enriched['tmdbId'] = pd.to_numeric(df_tmdb_enriched['tmdbId'], errors='raise').astype('Int64')
    assert df_tmdb_enriched['tmdbId'].is_unique
print('Metadati TMDb parsati:', len(df_tmdb_enriched))

Metadati TMDb parsati: 9622


## 6. Merge completo con schema originale

Questo blocco mantiene lo schema `ml_*`, `tmdb_*`, `text_combined`, ecc. del notebook 02. La modifica chiave è che la base è sempre costituita da **tutti** i film e non da un subset temporaneo. Le deduplicazioni silenziose sono sostituite da controlli espliciti.

In [23]:
print("\n" + "="*80)
print("MERGE E DATASET FINALE (SCHEMA CANONICO)")
print("="*80)

# Base MovieLens
df_ml_base = df_to_fetch[[
    'movieId', 'imdbId', 'tmdbId', 'title', 'genres'
]].copy()
df_ml_base['tmdbId'] = df_ml_base['tmdbId'].astype('Int64')
assert df_ml_base['movieId'].is_unique, 'La base MovieLens contiene movieId duplicati'

# # Anno MovieLens estratto dal suffisso del titolo, distinto dall'anno TMDb
# df_ml_base['ml_year'] = pd.to_numeric(
#     df_ml_base['title'].str.extract(r'\((\d{4})\)\s*$', expand=False),
#     errors='coerce',
# ).astype('Int64')

# Carica ratings per metriche MovieLens canoniche
ratings_path = DATA_RAW / 'ratings.csv'
df_ratings = pd.read_csv(ratings_path)
ml_ratings_stats = (
    df_ratings.groupby('movieId')
    .agg(
        ml_rating_mean=('rating', 'mean'),
        ml_rating_count=('rating', 'count'),
        ml_rating_std=('rating', 'std'),
        ml_unique_users=('userId', 'nunique'),
    )
    .reset_index()
)

# Weighted rating (stile IMDb): (v/(v+m))*R + (m/(v+m))*C
C = ml_ratings_stats['ml_rating_mean'].mean()
m = ml_ratings_stats['ml_rating_count'].quantile(0.60)
ml_ratings_stats['ml_weighted_rating'] = (
    (ml_ratings_stats['ml_rating_count'] / (ml_ratings_stats['ml_rating_count'] + m)) * ml_ratings_stats['ml_rating_mean']
    + (m / (ml_ratings_stats['ml_rating_count'] + m)) * C
)

# Carica tags per campi canonical MovieLens
tags_path = DATA_RAW / 'tags.csv'
df_tags = pd.read_csv(tags_path)

def build_tag_weights(group: pd.Series) -> str:
    counts = group.value_counts()
    return '|'.join([f"{tag}:{cnt}" for tag, cnt in counts.items()])

ml_tags_stats = (
    df_tags.groupby('movieId')
    .agg(
        ml_tag_application_count=('tag', 'count'),
        ml_unique_tag_count=('tag', 'nunique'),
        ml_tags=('tag', lambda s: ', '.join(sorted(set([str(x).strip() for x in s.dropna() if str(x).strip()])))),
        ml_tag_text=('tag', lambda s: ' '.join([str(x).strip().lower() for x in s.dropna() if str(x).strip()])),
        ml_tag_weights=('tag', build_tag_weights),
    )
    .reset_index()
)

# Rinominare colonne base MovieLens
df_ml_base = df_ml_base.rename(columns={
    'title': 'ml_title',
    'genres': 'ml_genres',
})

# Merge progressivo MovieLens + stats
df_ml_enriched = (
    df_ml_base
    .merge(ml_ratings_stats, on='movieId', how='left')
    .merge(ml_tags_stats, on='movieId', how='left')
)

# Merge con TMDb parsed
df_final_enriched = df_ml_enriched.merge(
    df_tmdb_enriched,
    on='tmdbId',
    how='left',
    suffixes=('', '_tmdb'),
    validate='many_to_one'
)

# Consolidamento imdbId: preferisci MovieLens, fallback TMDb
if 'imdbId_tmdb' in df_final_enriched.columns:
    df_final_enriched['imdbId'] = df_final_enriched['imdbId'].fillna(df_final_enriched['imdbId_tmdb'])
    df_final_enriched = df_final_enriched.drop(columns=['imdbId_tmdb'])

# Campi derivati testuali e canonical title
df_final_enriched['canonical_title'] = df_final_enriched['ml_title'].fillna(df_final_enriched.get('tmdb_title'))
df_final_enriched['text_movielens'] = (
    df_final_enriched[['ml_title', 'ml_genres', 'ml_tag_text']]
    .fillna('')
    .agg(' '.join, axis=1)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
if 'tmdb_text' not in df_final_enriched.columns:
    df_final_enriched['tmdb_text'] = (
        df_final_enriched[['overview', 'tagline', 'tmdb_keyword_text']]
        .fillna('')
        .agg(' '.join, axis=1)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

df_final_enriched['text_combined'] = (
    df_final_enriched[['text_movielens', 'tmdb_text']]
    .fillna('')
    .agg(' '.join, axis=1)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Status enrichment mancante
if 'tmdb_enrichment_status' in df_final_enriched.columns:
    df_final_enriched['tmdb_enrichment_status'] = df_final_enriched['tmdb_enrichment_status'].fillna('missing_tmdb')
else:
    df_final_enriched['tmdb_enrichment_status'] = 'missing_tmdb'

# Deduplicazione su movieId
assert df_final_enriched['movieId'].is_unique, 'Duplicate movieId dopo il merge: verificare il mapping TMDb'

# Ordine colonne schema canonico da struttura_campi_merged.txt
canonical_columns = [
    'movieId', 'imdbId', 'tmdbId',
    'canonical_title', 'ml_title', 'tmdb_title', 'tmdb_original_title',
    'release_date', 'release_year',
    'ml_genres', 'tmdb_genres',
    'ml_tags', 'ml_tag_weights', 'ml_tag_text', 'ml_tag_application_count', 'ml_unique_tag_count',
    'tmdb_keywords', 'tmdb_keyword_text', 'tmdb_keyword_count',
    'overview', 'tagline',
    'runtime', 'original_language', 'origin_countries', 'production_countries', 'spoken_languages', 'adult',
    'directors', 'writers', 'composers', 'cast_top', 'cast_characters_top',
    'collection_name', 'production_companies',
    'ml_rating_mean', 'ml_rating_count', 'ml_rating_std', 'ml_unique_users', 'ml_weighted_rating',
    'tmdb_vote_average', 'tmdb_vote_count', 'tmdb_popularity',
    'budget', 'revenue', 'poster_path', 'backdrop_path', 'homepage',
    'tmdb_enrichment_status', 'tmdb_retrieved_at',
    'text_movielens', 'tmdb_text', 'text_combined'
]


for col in canonical_columns:
    if col not in df_final_enriched.columns:
        df_final_enriched[col] = pd.NA

df_final_enriched = df_final_enriched[canonical_columns]

print(f"Base MovieLens: {len(df_ml_base)} film")
print(f"Dopo merge totale: {len(df_final_enriched)} film")
print(f"Colonne canoniche finali: {len(df_final_enriched.columns)}")

print("\nCampione dataset arricchito:")
print(df_final_enriched.head().to_string())

print("\nStatistiche dataset arricchito:")
print(f"  Film totali: {len(df_final_enriched)}")
print(f"  Con overview: {df_final_enriched['overview'].notna().sum()}")
print(f"  Con directors: {df_final_enriched['directors'].notna().sum()}")
print(f"  Con cast_top: {df_final_enriched['cast_top'].notna().sum()}")
print(f"  Con tmdb_keywords: {df_final_enriched['tmdb_keywords'].notna().sum()}")
print(f"  ml_rating_mean valorizzato: {df_final_enriched['ml_rating_mean'].notna().sum()}")
print(f"  tmdb_vote_average valorizzato: {df_final_enriched['tmdb_vote_average'].notna().sum()})" )


MERGE E DATASET FINALE (SCHEMA CANONICO)
Base MovieLens: 9742 film
Dopo merge totale: 9742 film
Colonne canoniche finali: 52

Campione dataset arricchito:
   movieId   imdbId  tmdbId                     canonical_title                            ml_title                   tmdb_title          tmdb_original_title release_date  release_year                                    ml_genres                           tmdb_genres                                          ml_tags                                        ml_tag_weights                                   ml_tag_text  ml_tag_application_count  ml_unique_tag_count                                                                                                                                                                                                                                                                                                  tmdb_keywords                                                                              

## 7. Testo GTE senza tag MovieLens


#### OLD -> preferisco mantenere retrocompatibilità

aprire blocco per dettagli descrizione

<!-- Il **dataset canonico e il merge restano invariati**. Il testo per embedding è una rappresentazione *derivata*, non il formato del dataset né una collection Qdrant.

La versione v2 della rappresentazione D usava `tmdb_title` al posto di `canonical_title` nel documento; per i film senza titolo TMDb si usa un fallback esplicito a `canonical_title`, per non lasciare il documento senza titolo. La lista dei campi parte dalla D del notebook 04, **escludendo soltanto `ml_tags`**. Le etichette e l'ordine sono coerenti con la D originale.

`text_combined` resta presente per compatibilità, e **contiene ancora i tag MovieLens**. Per creare gli embedding sul catalogo completo selezionare esplicitamente `gte_text_no_ml_tags` oppure (preferibilmente) riutilizzare nel modulo Qdrant la vera funzione di serializzazione `document__D_full_enriched_v2` con la sola rimozione dei tag, confrontando il testo risultante a parità di film. Questo notebook **non** crea né aggiorna collection Qdrant.

I tag MovieLens **restano nel dataset** (`ml_tags`, `ml_tag_text`, `ml_tag_weights`) ma non entrano nel testo `gte_text_no_ml_tags`. `text_combined` resta presente per compatibilità e **contiene tuttora i tag**: il notebook GTE/Qdrant dovrà consumare esplicitamente la nuova colonna.

La configurazione seguente replica la rappresentazione **D_full_enriched** osservata nel notebook 04, **meno** i tag MovieLens. Se l'attuale codice GTE usa campi diversi, va applicato lo stesso principio alla sua effettiva configurazione, senza modificare contemporaneamente altri fattori sperimentali. -->

In [ ]:
# # Base: rappresentazione D_full_enriched del notebook 04; v2 = titolo TMDb.
# # Cambiamento intenzionale rispetto alla D_full_enriched_v2: esclusione dei soli tag MovieLens.
# GTE_FIELDS_NO_TAGS = [
#     ('Title', 'embedding_title'),
#     ('genres', 'tmdb_genres'),
#     ('Overview', 'overview'),
#     ('Tagline', 'tagline'),
#     ('keywords', 'tmdb_keyword_text'),
#     ('Directors', 'directors'),
#     ('Writers', 'writers'),
#     ('Composers', 'composers'),
#     ('Main cast', 'cast_top'),
#     ('Main characters', 'cast_characters_top'),
#     ('Collection', 'collection_name'),
#     ('Production companies', 'production_companies'),
# ]

# # Mantieni i due titoli originali nel dataset, e definisci il titolo per l'embedding a valle del merge.
# # Non cambia canonical_title: è una feature derivata, usata solo dal documento GTE.
# tmdb_title_clean = df_final_enriched['tmdb_title'].astype('string').str.strip()
# tmdb_title_clean = tmdb_title_clean.mask(tmdb_title_clean.eq(''))
# df_final_enriched['embedding_title'] = tmdb_title_clean.fillna(
#     df_final_enriched['canonical_title'].astype('string').str.strip()
# )

# def clean_document_value(value):
#     if value is None: return ''
#     try:
#         if pd.isna(value): return ''
#     except (ValueError, TypeError):
#         pass
#     text = ' '.join(str(value).split()).strip()
#     return '' if text.lower() in {'', 'nan', 'none', 'null', '<na>'} else text

# def text_for_gte(row):
#     fragments = []
#     for label, column in GTE_FIELDS_NO_TAGS:
#         v = clean_document_value(row.get(column))
#         if v:
#             fragments.append(f'{label}: {v}')
#     return '\n'.join(fragments)

# df_final_enriched['gte_text_no_ml_tags'] = df_final_enriched.apply(text_for_gte, axis=1)
# assert df_final_enriched['gte_text_no_ml_tags'].str.strip().ne('').all(), 'Documenti GTE vuoti'
# print('Titoli con fallback:', int(tmdb_title_clean.isna().sum()))
# print('Esempio documento GTE v2-title senza tag:')
# print(df_final_enriched['gte_text_no_ml_tags'].iloc[0][:700])


Titoli con fallback: 119
Esempio documento GTE v2-title senza tag:
Title: Toy Story
genres: Family, Comedy, Animation, Adventure
Overview: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.
Tagline: Hang on for the comedy that goes to infinity and beyond!
keywords: rescue friendship mission jealousy villain bullying elementary school rivalry anthropomorphism friends computer animation buddy walkie talkie toy car nostalgic boy next door new toy neighborhood toy comes to life resourcefulness toy 3d animation jo


## 8. Controlli, copertura e export protetto

Il dataset viene esportato soltanto quando la quota di metadati TMDb caricati raggiunge la soglia configurata: in questo modo una prima prova da 50 film non sovrascrive un dataset usato per gli esperimenti su larga scala. Il catalogo mantiene comunque gli elementi MovieLens senza corrispondente TMDb.

In [26]:
assert len(df_final_enriched) == len(movies), 'Merge ha perso o duplicato film MovieLens'
assert df_final_enriched.movieId.is_unique, 'movieId duplicati'
assert 'ml_tags' in df_final_enriched.columns
# assert 'gte_text_no_ml_tags' in df_final_enriched.columns

known_ids = set(ml_base.tmdbId.dropna().astype(int))
coverage_ratio = len(known_ids & set(parsed_metadata)) / max(1, len(known_ids))
print(f'Film MovieLens conservati: {len(df_final_enriched)} / {len(movies)}')
print(f'TMDb ID parsati presenti: {len(known_ids & set(parsed_metadata))} / {len(known_ids)} ({coverage_ratio:.1%})')
print(f'Overview non vuoti: {df_final_enriched.overview.fillna("").astype(str).str.strip().ne("").sum()}')
# print(f'Testi GTE pronti: {df_final_enriched.gte_text_no_ml_tags.str.strip().ne("").sum()}')
print(f'TMDb status: {df_final_enriched.tmdb_enrichment_status.value_counts(dropna=False).to_dict()}')

if coverage_ratio < MIN_CACHE_COVERAGE_FOR_EXPORT:
    print(f'EXPORT NON ESEGUITO: copertura {coverage_ratio:.1%} < soglia {MIN_CACHE_COVERAGE_FOR_EXPORT:.0%}.')
    print('Completa il download (RUN_FETCH=True e MAX_NEW_MOVIES=None) oppure abbassa consapevolmente la soglia.')
else:
    base_path = DATA_PROCESSED / 'dataset_enriched_movielens_tmdb_full'
    df_final_enriched.to_csv(base_path.with_suffix('.csv'), index=False)
    try:
        df_final_enriched.to_parquet(base_path.with_suffix('.parquet'), index=False)
    except ImportError:
        logger.warning('pyarrow non presente: Parquet non disponibile, CSV già salvato.')
    manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'schema_version': 'movielens_tmdb_canonical_full',
        'movie_count': len(df_final_enriched),
        'tmdb_id_coverage_ratio': coverage_ratio,
        'tmdb_id_count': len(known_ids),
        'document_config': 'D_full_enriched_v2',
        'notes': (
            'Dataset MovieLens + TMDb completo. '
            'Schema canonico del notebook 02 mantenuto. '
            'I documenti D_full_enriched_v2 vengono costruiti '
            'a valle tramite MovieLensV2Adapter. '
            'I tag MovieLens sono conservati.'
        )
    }
    manifest_path = DATA_PROCESSED / 'dataset_enriched_movielens_tmdb_full_manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Salvato:', base_path.with_suffix('.csv'))
    print('Manifest:', manifest_path)

Film MovieLens conservati: 9742 / 9742
TMDb ID parsati presenti: 9622 / 9736 (98.8%)
Overview non vuoti: 9622
TMDb status: {'ok': 9623, 'missing_tmdb': 119}
Salvato: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/dataset_enriched_movielens_tmdb_full.csv
Manifest: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/dataset_enriched_movielens_tmdb_full_manifest.json


## 9. Passaggio alla valutazione A/B e a Qdrant

1. Per **GTE**, codificare `gte_text_no_ml_tags` con la versione del modello esatta e memorizzare `model_name`, `text_schema`, `max_length` e `dataset_version`.
2. Creare una **nuova collection Qdrant versionata** (es. `movies_gte_full_v2`), mantenendo la vecchia come riferimento. `movieId` deve essere l'ID stabile del point, non l'indice di riga del DataFrame.
3. Verificare che la `vector_size` della collection corrisponda all'output del modello GTE usato; effettuare gli upload in batch con controlli di conteggio.
4. Non usare `ml_rating_count`, `ml_rating_mean`, `tmdb_vote_average` o `tmdb_popularity` per produrre il vettore. Sono segnali utilizzabili solo nei baseline appropriati e con lo split corretto.
5. Per la valutazione di **popularity** e **matrix factorization**, effettuare lo split dei rating prima di calcolare gli score. Se si confrontano HR@10/NDCG@10, bloccare lo stesso insieme di utenti, ground truth, candidati negativi ed esclusioni per tutti i metodi.

**Limite di questa variante:** non contiene la pipeline GTE/Qdrant (che non è fra i file caricati) e non effettua chiamate reali TMDb nell'ambiente di questa risposta.